# Superstore Sales Analysis Using SQL



In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
from IPython.display import display


# STEP 1: LOAD DATASET


base_dir = Path(r"e:\celabal\Superstore_Sales_Analysis_SQL")
csv_path = base_dir / "dataset" / "Superstore.csv"

# Read CSV
raw_df = pd.read_csv(csv_path, encoding="cp1252")

# Standardize column names
raw_df.columns = [
    c.strip()
    .lower()
    .replace(" ", "_")
    .replace("-", "_")
    for c in raw_df.columns
]

# Convert dates
raw_df["order_date"] = pd.to_datetime(
    raw_df["order_date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

raw_df["ship_date"] = pd.to_datetime(
    raw_df["ship_date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")


# STEP 2: CREATE SQLITE DATABASE


conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")


# STEP 3: CREATE RAW TABLE


raw_df.to_sql(
    "superstore_raw",
    conn,
    if_exists="replace",
    index=False
)


# STEP 4: CREATE NORMALIZED TABLES


conn.executescript("""

DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;

CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    customer_name TEXT NOT NULL,
    segment TEXT,
    country TEXT,
    city TEXT,
    state TEXT,
    postal_code TEXT,
    region TEXT
);

CREATE TABLE products (
    product_id TEXT PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT,
    sub_category TEXT
);

CREATE TABLE orders (
    row_id INTEGER PRIMARY KEY,
    order_id TEXT,
    order_date TEXT,
    ship_date TEXT,
    ship_mode TEXT,
    customer_id TEXT,
    product_id TEXT,
    sales REAL,
    quantity INTEGER,
    discount REAL,
    profit REAL,

    FOREIGN KEY(customer_id)
        REFERENCES customers(customer_id),

    FOREIGN KEY(product_id)
        REFERENCES products(product_id)
);

""")


# STEP 5: INSERT DATA


# Customers

raw_df[
    [
        "customer_id",
        "customer_name",
        "segment",
        "country",
        "city",
        "state",
        "postal_code",
        "region"
    ]
].drop_duplicates(
    subset=["customer_id"]
).to_sql(
    "customers",
    conn,
    if_exists="append",
    index=False
)

# Products

raw_df[
    [
        "product_id",
        "product_name",
        "category",
        "sub_category"
    ]
].drop_duplicates(
    subset=["product_id"]
).to_sql(
    "products",
    conn,
    if_exists="append",
    index=False
)

# Orders

raw_df[
    [
        "row_id",
        "order_id",
        "order_date",
        "ship_date",
        "ship_mode",
        "customer_id",
        "product_id",
        "sales",
        "quantity",
        "discount",
        "profit"
    ]
].drop_duplicates(
    subset=["row_id"]
).to_sql(
    "orders",
    conn,
    if_exists="append",
    index=False
)


# STEP 6: DATABASE SUMMARY



print("DATABASE CREATED SUCCESSFULLY")


print(f"\nRows in superstore_raw: {len(raw_df):,}")


# STEP 7: SHOW ALL TABLES


print("\nAVAILABLE TABLES")

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """,
    conn
)

display(tables)

# STEP 8: SHOW ROW COUNTS


print("\nTABLE ROW COUNTS")

for table in [
    "superstore_raw",
    "customers",
    "products",
    "orders"
]:
    count_df = pd.read_sql_query(
        f"""
        SELECT COUNT(*) AS total_rows
        FROM {table};
        """,
        conn
    )

    print(f"\n{table}")
    display(count_df)


# STEP 9: DISPLAY SAMPLE RECORDS


print("\nCUSTOMERS TABLE")
display(
    pd.read_sql_query(
        "SELECT * FROM customers LIMIT 10;",
        conn
    )
)

print("\nPRODUCTS TABLE")
display(
    pd.read_sql_query(
        "SELECT * FROM products LIMIT 10;",
        conn
    )
)

print("\nORDERS TABLE")
display(
    pd.read_sql_query(
        "SELECT * FROM orders LIMIT 10;",
        conn
    )
)

print("\nSUPERSTORE_RAW TABLE")
display(
    pd.read_sql_query(
        "SELECT * FROM superstore_raw LIMIT 10;",
        conn
    )
)

def run_query(title, sql):
    result = pd.read_sql_query(sql, conn)
    display(result)
    return result

cte_window_results = {}

DATABASE CREATED SUCCESSFULLY

Rows in superstore_raw: 9,994

AVAILABLE TABLES


,name
0,superstore_raw
1,customers
2,products
3,orders



TABLE ROW COUNTS

superstore_raw


,total_rows
0,9994



customers


...


ORDERS TABLE


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,product_id,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,OFF-LA-10000240,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,OFF-ST-10000760,22.3680,2,0.20,2.5164
5,6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,FUR-FU-10001487,48.8600,7,0.00,14.1694
6,7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,OFF-AR-10002833,7.2800,4,0.00,1.9656
7,8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,TEC-PH-10002275,907.1520,6,0.20,90.7152
8,9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,OFF-BI-10003910,18.5040,3,0.20,5.7825
9,10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,OFF-AP-10002892,114.9000,5,0.00,34.4700



SUPERSTORE_RAW TABLE


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
5,6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7,0.00,14.1694
6,7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.2800,4,0.00,1.9656
7,8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.1520,6,0.20,90.7152
8,9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by S...,18.5040,3,0.20,5.7825
9,10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9000,5,0.00,34.4700



DATABASE READY FOR ANALYSIS


## Subquery Analysis

These cells answer the first five questions using subqueries.

In [14]:
subquery_results = {}

subqueries = [
    (
        "Q1 - Orders with sales greater than average sales",
        """
        SELECT *
        FROM orders
        WHERE sales > (SELECT AVG(sales) FROM orders)
        ORDER BY sales DESC
        """
    ),
    (
        "Q2 - Highest sales order for every customer",
        """
        SELECT o.*
        FROM orders o
        WHERE o.sales = (
            SELECT MAX(o2.sales)
            FROM orders o2
            WHERE o2.customer_id = o.customer_id
        )
        ORDER BY o.customer_id, o.sales DESC
        """
    ),
    (
        "Q3 - Customers whose sales exceed overall average customer sales",
        """
        WITH customer_totals AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_id, c.customer_name, ct.total_sales
        FROM customer_totals ct
        JOIN customers c ON c.customer_id = ct.customer_id
        WHERE ct.total_sales > (SELECT AVG(total_sales) FROM customer_totals)
        ORDER BY ct.total_sales DESC
        """
    ),
    (
        "Q4 - Products whose sales exceed average product sales",
        """
        WITH product_totals AS (
            SELECT product_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY product_id
        )
        SELECT p.product_id, p.product_name, pt.total_sales
        FROM product_totals pt
        JOIN products p ON p.product_id = pt.product_id
        WHERE pt.total_sales > (SELECT AVG(total_sales) FROM product_totals)
        ORDER BY pt.total_sales DESC
        """
    ),
    (
        "Q5 - Orders with profit greater than average profit",
        """
        SELECT *
        FROM orders
        WHERE profit > (SELECT AVG(profit) FROM orders)
        ORDER BY profit DESC
        """
    ),
]

for label, sql in subqueries:
    subquery_results[label] = run_query(label, sql)


Q1 - Orders with sales greater than average sales


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,product_id,sales,quantity,discount,profit
0,2698,CA-2014-145317,2014-03-18,2014-03-23,Standard Class,SM-20320,TEC-MA-10002412,22638.480,6,0.5,-1811.0784
1,6827,CA-2016-118689,2016-10-02,2016-10-09,Standard Class,TC-20980,TEC-CO-10004722,17499.950,5,0.0,8399.9760
2,8154,CA-2017-140151,2017-03-23,2017-03-25,First Class,RB-19360,TEC-CO-10004722,13999.960,4,0.0,6719.9808
3,2624,CA-2017-127180,2017-10-22,2017-10-24,First Class,TA-21385,TEC-CO-10004722,11199.968,4,0.2,3919.9888
4,4191,CA-2017-166709,2017-11-17,2017-11-22,Standard Class,HL-15040,TEC-CO-10004722,10499.970,3,0.0,5039.9856
...,...,...,...,...,...,...,...,...,...,...,...
2355,2615,CA-2014-147298,2014-04-26,2014-05-03,Standard Class,AG-10300,FUR-CH-10004886,230.280,3,0.2,23.0280
2356,3752,CA-2017-161956,2017-08-27,2017-08-29,Second Class,DR-12880,FUR-CH-10004886,230.280,3,0.2,23.0280
2357,4553,US-2014-106334,2014-12-27,2015-01-02,Standard Class,JF-15490,FUR-CH-10004886,230.280,3,0.2,23.0280
2358,7932,US-2016-168095,2016-07-15,2016-07-20,Standard Class,MC-17425,FUR-CH-10004886,230.280,3,0.2,23.0280



Q2 - Highest sales order for every customer


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,product_id,sales,quantity,discount,profit
0,5199,CA-2016-103982,2016-03-03,2016-03-08,Standard Class,AA-10315,OFF-SU-10000151,3930.072,3,0.2,-786.0144
1,2264,CA-2016-131065,2016-11-14,2016-11-16,Second Class,AA-10375,TEC-AC-10004145,499.980,2,0.0,114.9954
2,7706,CA-2016-114601,2016-08-26,2016-09-02,Standard Class,AA-10480,TEC-AC-10003911,479.970,3,0.0,163.1898
3,8010,CA-2015-110863,2015-11-17,2015-11-24,Standard Class,AA-10645,FUR-CH-10002073,1323.900,5,0.0,383.9310
4,8803,CA-2016-140935,2016-11-10,2016-11-12,First Class,AB-10015,FUR-BO-10003966,341.960,2,0.0,54.7136
...,...,...,...,...,...,...,...,...,...,...,...
790,9042,CA-2014-114335,2014-09-28,2014-10-03,Standard Class,XP-21865,FUR-FU-10000277,337.088,4,0.2,16.8544
791,3071,CA-2014-119375,2014-11-17,2014-11-22,Standard Class,YC-21895,OFF-ST-10002011,2934.330,7,0.0,792.2691
792,6210,CA-2017-119809,2017-08-18,2017-08-25,Standard Class,YS-21880,OFF-BI-10003925,2793.528,9,0.2,942.8157
793,5189,CA-2015-115567,2015-09-13,2015-09-18,Standard Class,ZC-21910,FUR-CH-10000015,1516.200,7,0.0,394.2120



Q3 - Customers whose sales exceed overall average customer sales


,customer_id,customer_name,total_sales
0,SM-20320,Sean Miller,25043.050
1,TC-20980,Tamara Chand,19052.218
2,RB-19360,Raymond Buch,15117.339
3,TA-21385,Tom Ashbrook,14595.620
4,AB-10105,Adrian Barton,14473.571
...,...,...,...
289,JK-16120,Julie Kriz,2932.484
290,SW-20455,Shaun Weien,2921.544
291,ML-17410,Maris LaWare,2921.500
292,RD-19585,Rob Dowd,2912.894



Q4 - Products whose sales exceed average product sales


,product_id,product_name,total_sales
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61599.824
1,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.384
2,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,22638.480
3,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,21870.576
4,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,19823.479
...,...,...,...
442,TEC-PH-10001809,Panasonic KX T7736-B Digital phone,1259.580
443,TEC-AC-10001539,Logitech G430 Surround Sound Gaming Headset wi...,1247.844
444,OFF-AP-10000026,Tripp Lite Isotel 6 Outlet Surge Protector wit...,1243.788
445,FUR-CH-10003846,Hon Valutask Swivel Chairs,1242.054



Q5 - Orders with profit greater than average profit


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,product_id,sales,quantity,discount,profit
0,6827,CA-2016-118689,2016-10-02,2016-10-09,Standard Class,TC-20980,TEC-CO-10004722,17499.950,5,0.0,8399.9760
1,8154,CA-2017-140151,2017-03-23,2017-03-25,First Class,RB-19360,TEC-CO-10004722,13999.960,4,0.0,6719.9808
2,4191,CA-2017-166709,2017-11-17,2017-11-22,Standard Class,HL-15040,TEC-CO-10004722,10499.970,3,0.0,5039.9856
3,9040,CA-2016-117121,2016-12-17,2016-12-21,Standard Class,AB-10105,OFF-BI-10000545,9892.740,13,0.0,4946.3700
4,4099,CA-2014-116904,2014-09-23,2014-09-28,Standard Class,SC-20095,OFF-BI-10001120,9449.950,5,0.0,4630.4755
...,...,...,...,...,...,...,...,...,...,...,...
2546,4792,CA-2015-104346,2015-12-11,2015-12-16,Standard Class,IM-15070,OFF-PA-10001166,85.056,3,0.2,28.7064
2547,5370,CA-2017-147403,2017-09-10,2017-09-13,First Class,KH-16630,OFF-PA-10003302,85.056,3,0.2,28.7064
2548,9881,CA-2015-104297,2015-05-29,2015-05-31,First Class,CC-12100,OFF-PA-10000474,85.056,3,0.2,28.7064
2549,6198,CA-2015-149909,2015-11-13,2015-11-17,Standard Class,RA-19915,OFF-PA-10000726,63.770,7,0.0,28.6965


## CTE and Window Function Analysis

These cells answer questions 6 through 20 using CTEs and window functions.

In [15]:
cte_window_results = {}

cte_window_queries = [
    (
        "Q6 - Total sales per customer",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_id, c.customer_name, cs.total_sales
        FROM customer_sales cs
        JOIN customers c ON c.customer_id = cs.customer_id
        ORDER BY cs.total_sales DESC
        """
    ),
    (
        "Q7 - Total profit per customer",
        """
        WITH customer_profit AS (
            SELECT customer_id, SUM(profit) AS total_profit
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_id, c.customer_name, cp.total_profit
        FROM customer_profit cp
        JOIN customers c ON c.customer_id = cp.customer_id
        ORDER BY cp.total_profit DESC
        """
    ),
    (
        "Q8 - Total quantity purchased by each customer",
        """
        WITH customer_quantity AS (
            SELECT customer_id, SUM(quantity) AS total_quantity
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_id, c.customer_name, cq.total_quantity
        FROM customer_quantity cq
        JOIN customers c ON c.customer_id = cq.customer_id
        ORDER BY cq.total_quantity DESC
        """
    ),
    (
        "Q9 - Customers whose sales are above average customer sales",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_id, c.customer_name, cs.total_sales
        FROM customer_sales cs
        JOIN customers c ON c.customer_id = cs.customer_id
        WHERE cs.total_sales > (SELECT AVG(total_sales) FROM customer_sales)
        ORDER BY cs.total_sales DESC
        """
    ),
    (
        "Q10 - Customers generating more than 10000 sales",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_id, c.customer_name, cs.total_sales
        FROM customer_sales cs
        JOIN customers c ON c.customer_id = cs.customer_id
        WHERE cs.total_sales > 10000
        ORDER BY cs.total_sales DESC
        """
    ),
    (
        "Q11 - Rank customers based on total sales",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_id, c.customer_name, cs.total_sales,
               RANK() OVER (ORDER BY cs.total_sales DESC) AS sales_rank
        FROM customer_sales cs
        JOIN customers c ON c.customer_id = cs.customer_id
        ORDER BY sales_rank, c.customer_name
        """
    ),
    (
        "Q12 - Row numbers for each order within customers",
        """
        SELECT o.*,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date, row_id) AS order_number_within_customer
        FROM orders o
        ORDER BY customer_id, order_date, row_id
        """
    ),
    (
        "Q13 - Top 3 customers",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        ), ranked_customers AS (
            SELECT customer_id, total_sales,
                   RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
            FROM customer_sales
        )
        SELECT c.customer_id, c.customer_name, rc.total_sales, rc.sales_rank
        FROM ranked_customers rc
        JOIN customers c ON c.customer_id = rc.customer_id
        WHERE rc.sales_rank <= 3
        ORDER BY rc.sales_rank, c.customer_name
        """
    ),
    (
        "Q14 - Top 10 customers",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        ), ranked_customers AS (
            SELECT customer_id, total_sales,
                   RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
            FROM customer_sales
        )
        SELECT c.customer_id, c.customer_name, rc.total_sales, rc.sales_rank
        FROM ranked_customers rc
        JOIN customers c ON c.customer_id = rc.customer_id
        WHERE rc.sales_rank <= 10
        ORDER BY rc.sales_rank, c.customer_name
        """
    ),
    (
        "Q15 - Highest sale order per customer using ROW_NUMBER()",
        """
        WITH ranked_orders AS (
            SELECT o.*,
                   ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY sales DESC, order_date DESC, row_id DESC) AS rn
            FROM orders o
        )
        SELECT *
        FROM ranked_orders
        WHERE rn = 1
        ORDER BY customer_id
        """
    ),
    (
        "Q16 - Cumulative sales",
        """
        SELECT o.order_date, o.row_id, o.order_id, o.sales,
               SUM(sales) OVER (ORDER BY order_date, row_id) AS cumulative_sales
        FROM orders o
        ORDER BY order_date, row_id
        """
    ),
    (
        "Q17 - Running profit",
        """
        SELECT o.order_date, o.row_id, o.order_id, o.profit,
               SUM(profit) OVER (ORDER BY order_date, row_id) AS running_profit
        FROM orders o
        ORDER BY order_date, row_id
        """
    ),
    (
        "Q18 - Dense rank based on customer sales",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_id, c.customer_name, cs.total_sales,
               DENSE_RANK() OVER (ORDER BY cs.total_sales DESC) AS sales_dense_rank
        FROM customer_sales cs
        JOIN customers c ON c.customer_id = cs.customer_id
        ORDER BY sales_dense_rank, c.customer_name
        """
    ),
    (
        "Q19 - Second highest customer sales",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        ), ranked AS (
            SELECT customer_id, total_sales,
                   DENSE_RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
            FROM customer_sales
        )
        SELECT c.customer_id, c.customer_name, r.total_sales
        FROM ranked r
        JOIN customers c ON c.customer_id = r.customer_id
        WHERE r.sales_rank = 2
        ORDER BY c.customer_name
        """
    ),
    (
        "Q20 - Third highest customer sales",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        ), ranked AS (
            SELECT customer_id, total_sales,
                   DENSE_RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
            FROM customer_sales
        )
        SELECT c.customer_id, c.customer_name, r.total_sales
        FROM ranked r
        JOIN customers c ON c.customer_id = r.customer_id
        WHERE r.sales_rank = 3
        ORDER BY c.customer_name
        """
    ),
]

for label, sql in cte_window_queries:
    cte_window_results[label] = run_query(label, sql)


Q6 - Total sales per customer


,customer_id,customer_name,total_sales
0,SM-20320,Sean Miller,25043.050
1,TC-20980,Tamara Chand,19052.218
2,RB-19360,Raymond Buch,15117.339
3,TA-21385,Tom Ashbrook,14595.620
4,AB-10105,Adrian Barton,14473.571
...,...,...,...
788,RS-19870,Roy Skaria,22.328
789,MG-18205,Mitch Gastineau,16.739
790,CJ-11875,Carl Jackson,16.520
791,LD-16855,Lela Donovan,5.304



Q7 - Total profit per customer


,customer_id,customer_name,total_profit
0,TC-20980,Tamara Chand,8981.3239
1,RB-19360,Raymond Buch,6976.0959
2,SC-20095,Sanjit Chand,5757.4119
3,HL-15040,Hunter Lopez,5622.4292
4,AB-10105,Adrian Barton,5444.8055
...,...,...,...
788,HG-14965,Henry Goldwyn,-2797.9635
789,SR-20425,Sharelle Roach,-3333.9144
790,LF-17185,Luke Foster,-3583.9770
791,GT-14635,Grant Thornton,-4108.6589



Q8 - Total quantity purchased by each customer


...

,customer_id,customer_name,total_sales,sales_dense_rank
0,SM-20320,Sean Miller,25043.050,1
1,TC-20980,Tamara Chand,19052.218,2
2,RB-19360,Raymond Buch,15117.339,3
3,TA-21385,Tom Ashbrook,14595.620,4
4,AB-10105,Adrian Barton,14473.571,5
...,...,...,...,...
788,RS-19870,Roy Skaria,22.328,789
789,MG-18205,Mitch Gastineau,16.739,790
790,CJ-11875,Carl Jackson,16.520,791
791,LD-16855,Lela Donovan,5.304,792



Q19 - Second highest customer sales


,customer_id,customer_name,total_sales
0,TC-20980,Tamara Chand,19052.218



Q20 - Third highest customer sales


,customer_id,customer_name,total_sales
0,RB-19360,Raymond Buch,15117.339


## Business, Profit, and Advanced Analysis

These cells answer questions 21 through 70 and include the final ranked customer report.

In [16]:
business_results = {}

customer_analysis = [
    (
        "Q21 - Top 5 customers",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_name, cs.total_sales
        FROM customer_sales cs
        JOIN customers c ON c.customer_id = cs.customer_id
        ORDER BY cs.total_sales DESC
        LIMIT 5
        """
    ),
    (
        "Q22 - Bottom 5 customers",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_name, cs.total_sales
        FROM customer_sales cs
        JOIN customers c ON c.customer_id = cs.customer_id
        ORDER BY cs.total_sales ASC
        LIMIT 5
        """
    ),
    (
        "Q23 - Customers who placed only one order",
        """
        SELECT c.customer_id, c.customer_name, COUNT(DISTINCT o.order_id) AS order_count
        FROM customers c
        JOIN orders o ON o.customer_id = c.customer_id
        GROUP BY c.customer_id, c.customer_name
        HAVING COUNT(DISTINCT o.order_id) = 1
        ORDER BY c.customer_name
        """
    ),
    (
        "Q24 - Customers who placed more than 10 orders",
        """
        SELECT c.customer_id, c.customer_name, COUNT(DISTINCT o.order_id) AS order_count
        FROM customers c
        JOIN orders o ON o.customer_id = c.customer_id
        GROUP BY c.customer_id, c.customer_name
        HAVING COUNT(DISTINCT o.order_id) > 10
        ORDER BY order_count DESC, c.customer_name
        """
    ),
    (
        "Q25 - Customers with above-average sales",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_name, cs.total_sales
        FROM customer_sales cs
        JOIN customers c ON c.customer_id = cs.customer_id
        WHERE cs.total_sales > (SELECT AVG(total_sales) FROM customer_sales)
        ORDER BY cs.total_sales DESC
        """
    ),
    (
        "Q26 - Customers generated highest profit",
        """
        WITH customer_profit AS (
            SELECT customer_id, SUM(profit) AS total_profit
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_name, cp.total_profit
        FROM customer_profit cp
        JOIN customers c ON c.customer_id = cp.customer_id
        ORDER BY cp.total_profit DESC
        LIMIT 1
        """
    ),
    (
        "Q27 - Customer purchased maximum quantity",
        """
        WITH customer_qty AS (
            SELECT customer_id, SUM(quantity) AS total_quantity
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_name, cq.total_quantity
        FROM customer_qty cq
        JOIN customers c ON c.customer_id = cq.customer_id
        ORDER BY cq.total_quantity DESC
        LIMIT 1
        """
    ),
    (
        "Q28 - Customer with highest order value",
        """
        SELECT c.customer_name, o.order_id, o.sales
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        ORDER BY o.sales DESC, o.order_date DESC
        LIMIT 1
        """
    ),
    (
        "Q29 - Customer with lowest order value",
        """
        SELECT c.customer_name, o.order_id, o.sales
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        ORDER BY o.sales ASC, o.order_date ASC
        LIMIT 1
        """
    ),
    (
        "Q30 - Customers who never generated profit",
        """
        SELECT c.customer_id, c.customer_name, SUM(o.profit) AS total_profit
        FROM customers c
        JOIN orders o ON o.customer_id = c.customer_id
        GROUP BY c.customer_id, c.customer_name
        HAVING SUM(o.profit) <= 0
        ORDER BY total_profit ASC, c.customer_name
        """
    ),
]

product_analysis = [
    (
        "Q31 - Top 5 products by sales",
        """
        SELECT p.product_id, p.product_name, SUM(o.sales) AS total_sales
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.product_id, p.product_name
        ORDER BY total_sales DESC
        LIMIT 5
        """
    ),
    (
        "Q32 - Bottom 5 products by sales",
        """
        SELECT p.product_id, p.product_name, SUM(o.sales) AS total_sales
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.product_id, p.product_name
        ORDER BY total_sales ASC
        LIMIT 5
        """
    ),
    (
        "Q33 - Top 5 products by profit",
        """
        SELECT p.product_id, p.product_name, SUM(o.profit) AS total_profit
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.product_id, p.product_name
        ORDER BY total_profit DESC
        LIMIT 5
        """
    ),
    (
        "Q34 - Most sold product",
        """
        SELECT p.product_id, p.product_name, SUM(o.quantity) AS total_quantity
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.product_id, p.product_name
        ORDER BY total_quantity DESC
        LIMIT 1
        """
    ),
    (
        "Q35 - Least sold product",
        """
        SELECT p.product_id, p.product_name, SUM(o.quantity) AS total_quantity
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.product_id, p.product_name
        ORDER BY total_quantity ASC
        LIMIT 1
        """
    ),
    (
        "Q36 - Highest revenue generating category",
        """
        SELECT p.category, SUM(o.sales) AS total_sales
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.category
        ORDER BY total_sales DESC
        LIMIT 1
        """
    ),
    (
        "Q37 - Lowest revenue generating category",
        """
        SELECT p.category, SUM(o.sales) AS total_sales
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.category
        ORDER BY total_sales ASC
        LIMIT 1
        """
    ),
    (
        "Q38 - Highest profit generating sub-category",
        """
        SELECT p.sub_category, SUM(o.profit) AS total_profit
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.sub_category
        ORDER BY total_profit DESC
        LIMIT 1
        """
    ),
    (
        "Q39 - Lowest profit generating sub-category",
        """
        SELECT p.sub_category, SUM(o.profit) AS total_profit
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.sub_category
        ORDER BY total_profit ASC
        LIMIT 1
        """
    ),
    (
        "Q40 - Products with above-average sales",
        """
        WITH product_sales AS (
            SELECT product_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY product_id
        )
        SELECT p.product_id, p.product_name, ps.total_sales
        FROM product_sales ps
        JOIN products p ON p.product_id = ps.product_id
        WHERE ps.total_sales > (SELECT AVG(total_sales) FROM product_sales)
        ORDER BY ps.total_sales DESC
        """
    ),
]

regional_analysis = [
    (
        "Q41 - Top performing region",
        """
        SELECT c.region, SUM(o.sales) AS total_sales
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        GROUP BY c.region
        ORDER BY total_sales DESC
        LIMIT 1
        """
    ),
    (
        "Q42 - Lowest performing region",
        """
        SELECT c.region, SUM(o.sales) AS total_sales
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        GROUP BY c.region
        ORDER BY total_sales ASC
        LIMIT 1
        """
    ),
    (
        "Q43 - Region with highest profit",
        """
        SELECT c.region, SUM(o.profit) AS total_profit
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        GROUP BY c.region
        ORDER BY total_profit DESC
        LIMIT 1
        """
    ),
    (
        "Q44 - Region with highest quantity sold",
        """
        SELECT c.region, SUM(o.quantity) AS total_quantity
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        GROUP BY c.region
        ORDER BY total_quantity DESC
        LIMIT 1
        """
    ),
    (
        "Q45 - State with highest sales",
        """
        SELECT c.state, SUM(o.sales) AS total_sales
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        GROUP BY c.state
        ORDER BY total_sales DESC
        LIMIT 1
        """
    ),
    (
        "Q46 - State with highest profit",
        """
        SELECT c.state, SUM(o.profit) AS total_profit
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        GROUP BY c.state
        ORDER BY total_profit DESC
        LIMIT 1
        """
    ),
    (
        "Q47 - Top 10 cities by sales",
        """
        SELECT c.city, SUM(o.sales) AS total_sales
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        GROUP BY c.city
        ORDER BY total_sales DESC
        LIMIT 10
        """
    ),
    (
        "Q48 - Top 10 cities by profit",
        """
        SELECT c.city, SUM(o.profit) AS total_profit
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        GROUP BY c.city
        ORDER BY total_profit DESC
        LIMIT 10
        """
    ),
]

profit_analysis = [
    ("Q49 - Total company profit", "SELECT SUM(profit) AS total_company_profit FROM orders"),
    ("Q50 - Average profit per order", "SELECT AVG(profit) AS average_profit_per_order FROM orders"),
    (
        "Q51 - Most profitable order",
        "SELECT order_id, row_id, sales, profit FROM orders ORDER BY profit DESC, sales DESC LIMIT 1"
    ),
    (
        "Q52 - Least profitable order",
        "SELECT order_id, row_id, sales, profit FROM orders ORDER BY profit ASC, sales ASC LIMIT 1"
    ),
    (
        "Q53 - Profit percentage by category",
        """
        SELECT p.category,
               SUM(o.profit) AS total_profit,
               SUM(o.sales) AS total_sales,
               CASE WHEN SUM(o.sales) = 0 THEN 0 ELSE ROUND((SUM(o.profit) * 100.0) / SUM(o.sales), 2) END AS profit_percentage
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.category
        ORDER BY profit_percentage DESC
        """
    ),
    (
        "Q54 - Loss-making products",
        """
        SELECT p.product_id, p.product_name, SUM(o.profit) AS total_profit
        FROM products p
        JOIN orders o ON o.product_id = p.product_id
        GROUP BY p.product_id, p.product_name
        HAVING SUM(o.profit) < 0
        ORDER BY total_profit ASC
        """
    ),
    (
        "Q55 - Loss-making customers",
        """
        SELECT c.customer_id, c.customer_name, SUM(o.profit) AS total_profit
        FROM customers c
        JOIN orders o ON o.customer_id = c.customer_id
        GROUP BY c.customer_id, c.customer_name
        HAVING SUM(o.profit) < 0
        ORDER BY total_profit ASC
        """
    ),
]

advanced_analysis = [
    (
        "Q56 - Monthly sales trend",
        """
        SELECT strftime('%Y-%m', order_date) AS sales_month,
               SUM(sales) AS monthly_sales
        FROM orders
        GROUP BY strftime('%Y-%m', order_date)
        ORDER BY sales_month
        """
    ),
    (
        "Q57 - Yearly sales trend",
        """
        SELECT strftime('%Y', order_date) AS sales_year,
               SUM(sales) AS yearly_sales
        FROM orders
        GROUP BY strftime('%Y', order_date)
        ORDER BY sales_year
        """
    ),
    (
        "Q58 - Monthly profit trend",
        """
        SELECT strftime('%Y-%m', order_date) AS profit_month,
               SUM(profit) AS monthly_profit
        FROM orders
        GROUP BY strftime('%Y-%m', order_date)
        ORDER BY profit_month
        """
    ),
    (
        "Q59 - Best sales month",
        """
        WITH monthly_sales AS (
            SELECT strftime('%Y-%m', order_date) AS month_start, SUM(sales) AS total_sales
            FROM orders
            GROUP BY strftime('%Y-%m', order_date)
        )
        SELECT month_start, total_sales
        FROM monthly_sales
        ORDER BY total_sales DESC
        LIMIT 1
        """
    ),
    (
        "Q60 - Worst sales month",
        """
        WITH monthly_sales AS (
            SELECT strftime('%Y-%m', order_date) AS month_start, SUM(sales) AS total_sales
            FROM orders
            GROUP BY strftime('%Y-%m', order_date)
        )
        SELECT month_start, total_sales
        FROM monthly_sales
        ORDER BY total_sales ASC
        LIMIT 1
        """
    ),
    (
        "Q61 - Average sales per month",
        """
        WITH monthly_sales AS (
            SELECT strftime('%Y-%m', order_date) AS month_start, SUM(sales) AS total_sales
            FROM orders
            GROUP BY strftime('%Y-%m', order_date)
        )
        SELECT AVG(total_sales) AS average_monthly_sales
        FROM monthly_sales
        """
    ),
    (
        "Q62 - Average profit per month",
        """
        WITH monthly_profit AS (
            SELECT strftime('%Y-%m', order_date) AS month_start, SUM(profit) AS total_profit
            FROM orders
            GROUP BY strftime('%Y-%m', order_date)
        )
        SELECT AVG(total_profit) AS average_monthly_profit
        FROM monthly_profit
        """
    ),
    (
        "Q63 - Running sales total",
        """
        SELECT order_date, row_id, sales,
               SUM(sales) OVER (ORDER BY order_date, row_id) AS running_sales_total
        FROM orders
        ORDER BY order_date, row_id
        """
    ),
    (
        "Q64 - Running profit total",
        """
        SELECT order_date, row_id, profit,
               SUM(profit) OVER (ORDER BY order_date, row_id) AS running_profit_total
        FROM orders
        ORDER BY order_date, row_id
        """
    ),
    (
        "Q65 - Top customer in every region",
        """
        WITH regional_customer_sales AS (
            SELECT c.region, c.customer_id, c.customer_name, SUM(o.sales) AS total_sales
            FROM orders o
            JOIN customers c ON c.customer_id = o.customer_id
            GROUP BY c.region, c.customer_id, c.customer_name
        ), ranked AS (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY region ORDER BY total_sales DESC) AS rn
            FROM regional_customer_sales
        )
        SELECT region, customer_name, total_sales
        FROM ranked
        WHERE rn = 1
        ORDER BY region
        """
    ),
    (
        "Q66 - Top product in every category",
        """
        WITH category_product_sales AS (
            SELECT p.category, p.product_id, p.product_name, SUM(o.sales) AS total_sales
            FROM orders o
            JOIN products p ON p.product_id = o.product_id
            GROUP BY p.category, p.product_id, p.product_name
        ), ranked AS (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY category ORDER BY total_sales DESC) AS rn
            FROM category_product_sales
        )
        SELECT category, product_name, total_sales
        FROM ranked
        WHERE rn = 1
        ORDER BY category
        """
    ),
    (
        "Q67 - Top city in every state",
        """
        WITH state_city_sales AS (
            SELECT c.state, c.city, SUM(o.sales) AS total_sales
            FROM orders o
            JOIN customers c ON c.customer_id = o.customer_id
            GROUP BY c.state, c.city
        ), ranked AS (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY state ORDER BY total_sales DESC) AS rn
            FROM state_city_sales
        )
        SELECT state, city, total_sales
        FROM ranked
        WHERE rn = 1
        ORDER BY state
        """
    ),
    (
        "Q68 - Highest order value in every category",
        """
        WITH category_order_values AS (
            SELECT p.category, o.order_id, o.sales
            FROM orders o
            JOIN products p ON p.product_id = o.product_id
        ), ranked AS (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY category ORDER BY sales DESC, order_id) AS rn
            FROM category_order_values
        )
        SELECT category, order_id, sales
        FROM ranked
        WHERE rn = 1
        ORDER BY category
        """
    ),
    (
        "Q69 - Highest profit order in every region",
        """
        WITH region_order_profit AS (
            SELECT c.region, o.order_id, o.profit
            FROM orders o
            JOIN customers c ON c.customer_id = o.customer_id
        ), ranked AS (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY region ORDER BY profit DESC, order_id) AS rn
            FROM region_order_profit
        )
        SELECT region, order_id, profit
        FROM ranked
        WHERE rn = 1
        ORDER BY region
        """
    ),
    (
        "Q70 - Customer contribution percentage to total sales",
        """
        WITH customer_sales AS (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        ), total_sales AS (
            SELECT SUM(total_sales) AS grand_total_sales
            FROM customer_sales
        )
        SELECT c.customer_id, c.customer_name, cs.total_sales,
               ROUND((cs.total_sales * 100.0) / t.grand_total_sales, 2) AS contribution_percentage
        FROM customer_sales cs
        JOIN customers c ON c.customer_id = cs.customer_id
        CROSS JOIN total_sales t
        ORDER BY contribution_percentage DESC
        """
    ),
]

for group in [customer_analysis, product_analysis, regional_analysis, profit_analysis, advanced_analysis]:
    for label, sql in group:
        business_results[label] = run_query(label, sql)

# Keep a few commonly inspected results in named variables
customer_top_5 = business_results["Q21 - Top 5 customers"]
region_top_10_cities = business_results["Q47 - Top 10 cities by sales"]
monthly_sales_trend = business_results["Q56 - Monthly sales trend"]


Q21 - Top 5 customers


,customer_name,total_sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571



Q22 - Bottom 5 customers


,customer_name,total_sales
0,Thais Sissman,4.833
1,Lela Donovan,5.304
2,Carl Jackson,16.520
3,Mitch Gastineau,16.739
4,Roy Skaria,22.328



Q23 - Customers who placed only one order


...

,category,order_id,sales
0,Furniture,CA-2017-118892,4416.174
1,Office Supplies,CA-2016-117121,9892.740
2,Technology,CA-2014-145317,22638.480



Q69 - Highest profit order in every region


,region,order_id,profit
0,Central,CA-2017-166709,5039.9856
1,East,CA-2017-140151,6719.9808
2,South,CA-2015-145352,3177.4750
3,West,CA-2016-118689,8399.9760



Q70 - Customer contribution percentage to total sales


,customer_id,customer_name,total_sales,contribution_percentage
0,SM-20320,Sean Miller,25043.050,1.09
1,TC-20980,Tamara Chand,19052.218,0.83
2,RB-19360,Raymond Buch,15117.339,0.66
3,TA-21385,Tom Ashbrook,14595.620,0.64
4,AB-10105,Adrian Barton,14473.571,0.63
...,...,...,...,...
788,RE-19405,Ricardo Emerson,48.360,0.00
789,RM-19750,Roland Murray,98.350,0.00
790,RS-19870,Roy Skaria,22.328,0.00
791,SG-20890,Susan Gilcrest,47.946,0.00


## Final Ranked Customer Report

This last cell produces the combined report with customer name, total sales, and rank.

In [17]:
final_report = run_query(
    "Final Report - Customer Name, Total Sales, Rank",
    """
    WITH customer_totals AS (
        SELECT customer_id, SUM(sales) AS total_sales
        FROM orders
        GROUP BY customer_id
    )
    SELECT c.customer_name AS "Customer Name",
           ct.total_sales AS "Total Sales",
           RANK() OVER (ORDER BY ct.total_sales DESC) AS "Rank"
    FROM customer_totals ct
    JOIN customers c ON c.customer_id = ct.customer_id
    ORDER BY 3, 1
    """
)

final_report


Final Report - Customer Name, Total Sales, Rank


,Customer Name,Total Sales,Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


,Customer Name,Total Sales,Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792
